# SOLUTION PROMPT 2

Solución especializada para predicción de churn.

- **Input**: EDA report de Prompt 2 Parte 1
- **Output**: AUC-ROC + comparación vs Prompt 1

Ejecuta cada celda en orden.

In [ ]:
# SETUP
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, roc_curve

print("✓ Librerías importadas")

In [ ]:
# CARGAR DATOS
with open("../datos/customer_churn_data.json", "r") as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(f"Dataset: {df.shape[0]} filas, {df.shape[1]} columnas")
print(f"\nChurn distribution:\n{df['churn_90d'].value_counts()}")
print(f"\nDesbalance: {df['churn_90d'].value_counts(normalize=True).round(3)}")

In [ ]:
# PREPARACIÓN: BASADA EN EDA

X = df.drop(columns=["customer_id", "churn_90d"])
y = df["churn_90d"]

# 1. Encoding categóricas
le = LabelEncoder()
X["product_type"] = le.fit_transform(X["product_type"])

# 2. Tratamiento de nulos (imputación por mediana)
numeric_cols = X.select_dtypes(include=[np.number]).columns
X[numeric_cols] = X[numeric_cols].fillna(X[numeric_cols].median())

# 3. Outliers: Capping en account_balance
q1 = X['account_balance'].quantile(0.25)
q3 = X['account_balance'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 3 * iqr
upper_bound = q3 + 3 * iqr
X['account_balance'] = X['account_balance'].clip(lower_bound, upper_bound)

print(f"✓ Datos preparados: {X.shape[1]} features")
print(f"Nulos restantes: {X.isna().sum().sum()}")

In [ ]:
# SPLIT CON ESTRATIFICACIÓN
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} ({y_train.value_counts()[0]} neg, {y_train.value_counts()[1]} pos)")
print(f"Test:  {X_test.shape[0]} ({y_test.value_counts()[0]} neg, {y_test.value_counts()[1]} pos)")

In [ ]:
# ESCALADO
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✓ Datos escalados")

In [ ]:
# MODELO: GradientBoosting (mejor para ranking con desbalance)
model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    subsample=0.8
)

model.fit(X_train_scaled, y_train)
print("✓ Modelo entrenado")

In [ ]:
# EVALUACIÓN
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
y_pred = model.predict(X_test_scaled)

auc = roc_auc_score(y_test, y_pred_proba)

print("="*60)
print(f"AUC-ROC: {auc:.4f}")
print("="*60)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
# TOP 10% - RANKING (MÉTRICA DE NEGOCIO)
top_10_idx = np.argsort(-y_pred_proba)[:len(y_test)//10]
precision_top10 = y_test.iloc[top_10_idx].mean()
recall_top10 = y_test.iloc[top_10_idx].sum() / y_test.sum()

print(f"\nTOP 10% ANALYSIS:")
print(f"Precisión en top 10%: {precision_top10:.3f} (% de churners en contactos)")
print(f"Recall en top 10%: {recall_top10:.3f} (% de churners detectados)")

In [ ]:
# VISUALIZACIÓN: ROC CURVE
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'AUC = {auc:.3f}', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Churn Prediction')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("✓ Gráfico guardado")

In [ ]:
# GUARDAR AUC
with open("./auc_prompt_2.txt", "w") as f:
    f.write(f"{auc:.4f}")

print(f"✓ AUC guardado en: ./auc_prompt_2.txt")
print(f"\nValor: {auc:.4f}")

In [ ]:
# COMPARACIÓN: PROMPT 1 vs PROMPT 2
try:
    with open("../prueba_prompt_1/auc_prompt_1.txt", "r") as f:
        auc_1 = float(f.read().strip())
    
    print("\n" + "="*60)
    print("COMPARACIÓN: ENFOQUE GENÉRICO vs ESPECIALIZADO")
    print("="*60)
    print(f"Prompt 1 (Genérico):       AUC = {auc_1:.4f}")
    print(f"Prompt 2 (Especializado):  AUC = {auc:.4f}")
    print(f"\nMejoría: +{(auc - auc_1):.4f} ({100*(auc - auc_1)/auc_1:.2f}%)")
    print("="*60)
except FileNotFoundError:
    print("⚠ No se encontró auc_prompt_1.txt - ejecuta Prompt 1 primero")